# 03 — XGBoost Baseline with SHAP Explanations

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import yaml
import shap

with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

from src.data.split import load_splits
from src.models.xgboost_baseline import train_xgboost, extract_shap_values
from src.evaluation.metrics import compute_all_metrics

## 1. Load Data & Train

In [ ]:
splits = load_splits(config['data']['processed_dir'])
X_train = splits['X_train'].values
X_cal = splits['X_cal'].values
X_test = splits['X_test'].values
y_train = splits['y_train']
y_cal = splits['y_cal']
y_test = splits['y_test']
feature_names = splits['feature_names']

model = train_xgboost(X_train, y_train, X_cal, y_cal, config['xgboost'])
y_prob = model.predict_proba(X_test)[:, 1]

metrics = compute_all_metrics(y_test, y_prob)
for k, v in metrics.items():
    print(f'{k:>15s}: {v:.4f}')

## 2. SHAP Analysis

In [ ]:
shap_data = extract_shap_values(model, X_test, feature_names)

# Summary plot
shap.summary_plot(shap_data['shap_values'], X_test, feature_names=feature_names)

In [ ]:
# SHAP dependence plots for top features
mean_abs_shap = np.abs(shap_data['shap_values']).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, idx in enumerate(top_idx):
    ax = axes[i // 3, i % 3]
    ax.scatter(X_test[:, idx], shap_data['shap_values'][:, idx], 
              alpha=0.3, s=3, color='#FF5722')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel(feature_names[idx])
    ax.set_ylabel('SHAP value')
    ax.set_title(feature_names[idx])
plt.tight_layout()
plt.show()

## 3. Feature Importance

In [ ]:
importance = mean_abs_shap
order = np.argsort(importance)[::-1]

print('Feature importance (mean |SHAP|):')
for i in order:
    print(f'  {feature_names[i]:>12s}: {importance[i]:.6f}')